# Modélisation du Système de Recommandation
Dans ce notebook, nous allons développer et comparer 4 approches (fonctions de recommandation d'articles) pour notre système de recommandation :

0. **Calcul de la popularité**
1. **Content-Based (Filtrage basé sur le contenu)**
2. **Collaborative Filtering (Filtrage collaboratif)**
3. **Approche Hybride**

L'objectif est de créer une fonction prenant en entrée un `user_id` et retournant 5 `article_id` recommandés.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings('ignore')

print("Librairies importées avec succès.")

Librairies importées avec succès.


In [5]:
import scipy.sparse as sparse

# Définition des chemins
DATA_DIR = "../Data/news-portal-user-interactions-by-globocom"
GENERATED_DIR = "../Generated"

METADATA_PATH = os.path.join(DATA_DIR, "articles_metadata.csv")
CLICKS_SAMPLE_PATH = os.path.join(DATA_DIR, "clicks_sample.csv") # On peut le garder pour des tests rapides
EMBEDDINGS_PCA_PATH = os.path.join(GENERATED_DIR, "articles_embeddings_pca.pickle")
POPULARITY_PATH = os.path.join(GENERATED_DIR, "Popularity_data/articles_popularity_by_clicks.parquet")
SPARSE_MATRIX_PATH = os.path.join(GENERATED_DIR, "sparse_item_user_matrix.npz")

# Chargement des données
print("Chargement des métadonnées des articles...")
articles_df = pd.read_csv(METADATA_PATH)
print("Chargement de l'échantillon des clics...")
clicks_df = pd.read_csv(CLICKS_SAMPLE_PATH)

print("Chargement des embeddings (Content-Based)...")
with open(EMBEDDINGS_PCA_PATH, 'rb') as f:
    embeddings_pca = pickle.load(f)

print("Chargement de la popularité (Cold Start)...")
popularity_df = pd.read_parquet(POPULARITY_PATH)

print("Chargement de la matrice creuse et mappings (Collaborative Filtering)...")
sparse_item_user = sparse.load_npz(SPARSE_MATRIX_PATH)
with open(os.path.join(GENERATED_DIR, "Mapping_data/user_mapping.pkl"), 'rb') as f:
    user_mapping = pickle.load(f)
with open(os.path.join(GENERATED_DIR, "Mapping_data/item_mapping.pkl"), 'rb') as f:
    item_mapping = pickle.load(f)

print("✅ Chargement terminé !")


Chargement des métadonnées des articles...
Chargement de l'échantillon des clics...
Chargement des embeddings (Content-Based)...
Chargement de la popularité (Cold Start)...
Chargement de la matrice creuse et mappings (Collaborative Filtering)...
✅ Chargement terminé !


# Liste des articles par utilisateur

In [6]:
def get_user_articles(user_id, clicks_df):
    """
    Retourne la liste des article_id lus par un utilisateur.
    """
    return clicks_df[clicks_df['user_id'] == user_id]['click_article_id'].tolist()

# 1. On prend un utilisateur au hasard (ici, le tout premier de notre dataset)
sample_user_id = clicks_df['user_id'].iloc[0]

# 2. On appelle la fonction
articles_lus = get_user_articles(sample_user_id, clicks_df)

# 3. On affiche le résultat
print(f"L'utilisateur {sample_user_id} a lu {len(articles_lus)} article(s) :")
print(articles_lus)


L'utilisateur 0 a lu 2 article(s) :
[157541, 68866]


# Metadonnées des articles 
(pour l'affichage des titres d'articles dans les résultats)

In [11]:
def get_article_metadata(article_ids, articles_df):
    """
    Récupère les informations disponibles sur les articles (catégorie, nombre de mots, etc.).
    NOTE: Le dataset Globocom est anonymisé, il n'y a donc pas de vrais 'titres' ou 'libellés' d'articles.
    """
    return articles_df[articles_df['article_id'].isin(article_ids)]

# 0. Popularité

In [7]:
# fonction qui retourne les articles les plus populaires en tant que recommandation
def recommend_popular(popularity_df, n_recommendations=5):
    """
    Retourne les N articles les plus populaires.
    C'est notre solution de secours (Fallback) pour le Cold Start.
    """
    # Le DataFrame popularity_df a déjà été trié par popularité lors de l'EDA
    popular_articles = popularity_df['click_article_id'].head(n_recommendations).tolist()
    
    return popular_articles

# --- Test de la fonction ---
print(f"Top {5} des articles les plus populaires :")
print(recommend_popular(popularity_df, n_recommendations=5))


Top 5 des articles les plus populaires :
[160974, 272143, 336221, 234698, 123909]


# 1. Approche Collaborative filtring

In [9]:
# fonction qui retourne les articles recommandés avec ALS
import implicit

print("Entraînement du modèle ALS en cours...")

# 1. On prépare la matrice User-Item avant l'entraînement (Lignes = Users, Colonnes = Articles)
sparse_user_item = sparse_item_user.T.tocsr() 

# 2. Initialisation du modèle ALS
model_als = implicit.als.AlternatingLeastSquares(factors=50, iterations=15, regularization=0.1, random_state=42)

# 3. Entraînement sur la matrice User-Item
model_als.fit(sparse_user_item)
print("✅ Modèle ALS entraîné avec succès !")

# 4. Inversion des dictionnaires pour passer de (Vrai ID <-> Code interne de la matrice)
user_id_to_code = {v: k for k, v in user_mapping.items()}
item_code_to_id = {k: v for k, v in item_mapping.items()}

def recommend_collaborative(user_id, model=model_als, n_recommendations=5):
    """
    Génère des recommandations basées sur le filtrage collaboratif (ALS).
    """
    # Gestion du Cold Start
    if user_id not in user_id_to_code:
        print(f"⚠️ Utilisateur {user_id} inconnu (Cold Start). Renvoi de la popularité.")
        return recommend_popular(popularity_df, n_recommendations)
        
    user_code = user_id_to_code[user_id]
    
    # Prédiction avec ALS
    # N'oublie pas de préciser [user_code:user_code+1] pour éviter un warning d'implicit
    recommended_codes, scores = model.recommend(
        user_code, 
        sparse_user_item[user_code], 
        N=n_recommendations,
        filter_already_liked_items=True 
    )
    
    # Convertir les codes numpy internes vers les vrais article_id
    recommendations = [item_code_to_id[int(code)] for code in recommended_codes]
    
    return recommendations


# --- Test de la fonction ---
sample_user_id = clicks_df['user_id'].iloc[0]
print(f"\nTop {5} recommandations Collaboratives (ALS) pour l'utilisateur {sample_user_id} :")
print(recommend_collaborative(sample_user_id))


Entraînement du modèle ALS en cours...


100%|██████████| 15/15 [00:13<00:00,  1.13it/s]

✅ Modèle ALS entraîné avec succès !

Top 5 recommandations Collaboratives (ALS) pour l'utilisateur 0 :
[293114, 284985, 272218, 160940, 124749]


# 2. Approche Content-Based (Basée sur le contenu)
L'idée ici est de recommander des articles qui sont sémantiquement similaires à ceux que l'utilisateur a déjà lus.

**Étapes :**
1. Identifier les articles lus par un utilisateur cible (`user_id`).
2. Récupérer les embeddings de ces articles lus.
3. Calculer le 'profil' de l'utilisateur (par exemple, la moyenne des embeddings des articles qu'il a lus).
4. Calculer la similarité cosinus entre ce profil utilisateur et tous les autres articles disponibles.
5. Retourner les 5 articles les plus similaires (que l'utilisateur n'a pas encore lus).

In [12]:
def recommend_content_based(user_id, clicks_df, embeddings, n_recommendations=5):
    """
    Génère des recommandations basées sur le contenu pour un utilisateur donné.
    """
    # 1. Obtenir les articles lus par l'utilisateur
    read_articles = get_user_articles(user_id, clicks_df)
    
    # 2. Gestion du Cold Start (utilisateur sans historique)
    if not read_articles:
        print(f"Utilisateur {user_id} sans historique. Renvoi des articles les plus populaires par défaut.")
        popular_articles = clicks_df['click_article_id'].value_counts().head(n_recommendations).index.tolist()
        return popular_articles
    
    # 3. Calculer le profil utilisateur (moyenne des embeddings des articles lus)
    # On s'assure que l'ID de l'article ne dépasse pas la taille de la matrice d'embeddings
    valid_articles = [art_id for art_id in read_articles if art_id < embeddings.shape[0]]
    
    if not valid_articles:
        return []
        
    user_profile = np.mean(embeddings[valid_articles], axis=0).reshape(1, -1)
    
    # 4. Calculer la similarité cosinus entre le profil et tous les embeddings
    similarities = cosine_similarity(user_profile, embeddings)[0]
    
    # 5. Trier et retourner le top N (en excluant les articles déjà lus)
    # Obtenir les indices triés par ordre décroissant de similarité
    similar_indices = np.argsort(similarities)[::-1]
    
    recommendations = []
    for idx in similar_indices:
        if idx not in read_articles:
            recommendations.append(int(idx))
        if len(recommendations) == n_recommendations:
            break
            
    return recommendations


# Test de la fonction de recommandation
sample_user_id = clicks_df['user_id'].iloc[0]
print(f"Recommandations pour l'utilisateur {sample_user_id} :")
recs = recommend_content_based(sample_user_id, clicks_df, embeddings_pca, n_recommendations=5)
print(f"IDs recommandés : {recs}\n")

print("Détails des articles recommandés (métadonnées) : ")
display(get_article_metadata(recs, articles_df))


Recommandations pour l'utilisateur 0 :
IDs recommandés : [157519, 157944, 159495, 156690, 162856]

Détails des articles recommandés (métadonnées) : 


,article_id,category_id,created_at_ts,publisher_id,words_count
156690,156690,281,1496859579000,0,170
157519,157519,281,1506064952000,0,272
157944,157944,281,1512504667000,0,204
159495,159495,281,1506021561000,0,261
162856,162856,281,1511000684000,0,237


# 3. Approche Hybride

In [14]:
def recommend_hybrid(user_id, n_recommendations=5):
    """
    Combine les prédictions avec des pondérations spécifiques :
    - 40% Collaborative Filtering (ALS)
    - 40% Content-Based
    - 20% Popularité
    """
    # Définition de l'importance (poids) de chaque technique
    w_cf = 0.40
    w_cb = 0.40
    w_pop = 0.20
    
    # On génère un plus grand bassin de recommandations (ex: 15) pour pouvoir bien les mélanger
    n_pool = n_recommendations * 3 
    
    # 1. Récupération des listes de chaque modèle
    cf_recs = recommend_collaborative(user_id, model_als, n_recommendations=n_pool)
    cb_recs = recommend_content_based(user_id, clicks_df, embeddings_pca, n_recommendations=n_pool)
    pop_recs = recommend_popular(popularity_df, n_recommendations=n_pool)
    
    # Dictionnaire pour stocker le score total de chaque article
    scores = {}
    
    # 2. Points pour le Collaborative Filtering (40%)
    for rank, article_id in enumerate(cf_recs):
        points = n_pool - rank # Le 1er de la liste gagne le max de points, le dernier gagne 1 point
        scores[article_id] = scores.get(article_id, 0) + (points * w_cf)
        
    # 3. Points pour le Content-Based (40%)
    for rank, article_id in enumerate(cb_recs):
        points = n_pool - rank
        scores[article_id] = scores.get(article_id, 0) + (points * w_cb)
        
    # 4. Points pour la Popularité (20%)
    for rank, article_id in enumerate(pop_recs):
        points = n_pool - rank
        scores[article_id] = scores.get(article_id, 0) + (points * w_pop)
        
    # 5. Tri des articles par score global décroissant
    sorted_articles = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    
    # On retourne uniquement les IDs du Top N
    return [article_id for article_id, score in sorted_articles][:n_recommendations]

# --- Test ---
sample_user_id = clicks_df['user_id'].iloc[0]
print(f"Top {5} recommandations Hybrides (Pondérées) pour l'utilisateur {sample_user_id} :")
print(recommend_hybrid(sample_user_id))


Top 5 recommandations Hybrides (Pondérées) pour l'utilisateur 0 :
[293114, 157519, 284985, 157944, 272218]


# 4. Évaluation des modèles

Explication de la méthode : 

On va isoler les utilisateurs qui ont cliqué au moins 2 fois. On cache leur tout dernier clic (ce sera notre "Test"), et on demande aux modèles d'essayer de le deviner. 
On mesurera deux choses cruciales pour le déploiement sur Azure :

**Hit Ratio @ 5 :** Le pourcentage de fois où l'article cible était bien dans le Top 5 recommandé.
**Latence :** Le temps (en millisecondes) que met la fonction à répondre.

**Remarque :** Dans ce code simplifié pour l'MVP, le modèle ALS a "triché" car il a été entraîné sur la matrice globale. Pour une évaluation parfaite, il aurait fallu reconstruire une matrice amputée des clics de test et réentraîner ALS. Mais cela suffit pour voir l'ordre de grandeur !



In [18]:
import time
import random
import scipy.sparse as sparse

print("🛠️ 1. Préparation du vrai Train/Test Split...")
# On isole les utilisateurs valides (>= 2 clics)
user_counts = clicks_df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 2].index.tolist()

# On prend un échantillon de 100 utilisateurs
random.seed(42)
test_users_sample = random.sample(valid_users, 100)

# Le "Test Set" (la cible) est le TOUT DERNIER clic de notre échantillon
df_sample = clicks_df[clicks_df['user_id'].isin(test_users_sample)].sort_values('click_timestamp')
test_set = df_sample.groupby('user_id').tail(1)

# Le "Train Set" est TOUT LE RESTE (clicks_df moins test_set)
train_set = clicks_df.drop(test_set.index)

print("⚙️ 2. Re-création de la matrice sans la triche...")
interactions_train = train_set.groupby(['user_id', 'click_article_id']).size().reset_index(name='clicks')

# On réutilise nos dictionnaires de mapping inverses
u_dict = {v: k for k, v in user_mapping.items()}
i_dict = {v: k for k, v in item_mapping.items()}

user_codes = interactions_train['user_id'].map(u_dict)
item_codes = interactions_train['click_article_id'].map(i_dict)

# On filtre les éventuelles erreurs
valid = user_codes.notna() & item_codes.notna()
user_codes, item_codes, clicks_val = user_codes[valid].astype(int), item_codes[valid].astype(int), interactions_train.loc[valid, 'clicks'].astype(float)

# Matrice Train uniquement
sparse_item_user_train = sparse.csr_matrix((clicks_val, (item_codes, user_codes)), shape=sparse_item_user.shape)
sparse_user_item_train = sparse_item_user_train.T.tocsr()

print("🧠 3. Ré-entraînement du modèle ALS sur le Train Set...")
model_als_eval = implicit.als.AlternatingLeastSquares(factors=50, iterations=15, regularization=0.1, random_state=42)
model_als_eval.fit(sparse_user_item_train)

print("\n🚀 4. Lancement de l'évaluation propre !")
hits = {'Popularité': 0, 'Content-Based': 0, 'Collaboratif (ALS)': 0}
times = {'Popularité': 0.0, 'Content-Based': 0.0, 'Collaboratif (ALS)': 0.0}

for _, row in test_set.iterrows():
    u_id = row['user_id']
    target = row['click_article_id']
    u_code = u_dict[u_id]
    
    # --- Pop ---
    t0 = time.time()
    recs_pop = recommend_popular(popularity_df, 5)
    times['Popularité'] += (time.time() - t0)
    if target in recs_pop: hits['Popularité'] += 1
        
    # --- CB ---
    t0 = time.time()
    # On passe bien "train_set" et pas "clicks_df" !
    recs_cb = recommend_content_based(u_id, train_set, embeddings_pca, 5) 
    times['Content-Based'] += (time.time() - t0)
    if target in recs_cb: hits['Content-Based'] += 1
        
    # --- CF (ALS) on bypass notre fonction pour utiliser la matrice Train ---
    t0 = time.time()
    rec_codes, _ = model_als_eval.recommend(u_code, sparse_user_item_train[u_code], N=5, filter_already_liked_items=True)
    recs_cf = [item_mapping[c] for c in rec_codes]
    times['Collaboratif (ALS)'] += (time.time() - t0)
    if target in recs_cf: hits['Collaboratif (ALS)'] += 1

print("\n📊 RÉSULTATS DE L'ÉVALUATION (Hit Ratio @ 5) sur 100 users :")
for model, h in hits.items():
    print(f"- {model} : {h}% de réussite")

print("\n⏱️ TEMPS DE RÉPONSE MOYEN PAR RECOMMANDATION :")
for model, t in times.items():
    print(f"- {model} : {(t / 100) * 1000:.2f} ms")


🛠️ 1. Préparation du vrai Train/Test Split...
⚙️ 2. Re-création de la matrice sans la triche...
🧠 3. Ré-entraînement du modèle ALS sur le Train Set...


100%|██████████| 15/15 [00:00<00:00, 33.07it/s]



🚀 4. Lancement de l'évaluation propre !

📊 RÉSULTATS DE L'ÉVALUATION (Hit Ratio @ 5) sur 100 users :
- Popularité : 0% de réussite
- Content-Based : 1% de réussite
- Collaboratif (ALS) : 4% de réussite

⏱️ TEMPS DE RÉPONSE MOYEN PAR RECOMMANDATION :
- Popularité : 0.08 ms
- Content-Based : 36.58 ms
- Collaboratif (ALS) : 0.38 ms
